# Capstone Project 02 — Deduction Identification

Implements the acceptance criteria from the Deduction Identification brief: identify short
payments, capture the deduction reason from remittance remarks, assign a reason code, apply
partial cash (close the paid portion, leave the deduction open), auto-write-off differences
within tolerance, and route anything unresolved for follow-up.

This notebook reuses `foundry_helper.py` (same module as Capstone 01). Reason classification is
**keyword-first** (deterministic, auditable) and only falls back to the deployed Azure AI Foundry
`gpt-5` agent when the remittance remark is free-text that doesn't cleanly match a keyword — a
realistic use of an LLM: handle the messy, unstructured minority of cases without giving up
determinism for the clean majority.

## 1. Setup

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dataclasses import dataclass
from datetime import date
import itertools
import json
import re

import pandas as pd

from foundry_helper import ask_agent_text, MODEL_DEPLOYMENT_NAME, MCP_SERVER_NAME

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

print(f"Model deployment  : {MODEL_DEPLOYMENT_NAME}")
print(f"MCP tool attached : {MCP_SERVER_NAME}")

Model deployment  : gpt-5
MCP tool attached : Microsoft Learn MCP server


## 2. Reference data — supported reasons & reason codes

Straight from the brief.

In [3]:
REASON_CODE_MAP = {
    "Pricing Issue": "D01",
    "Freight Claim": "D02",
    "Damage Claim": "D03",
    "Tax Difference": "D04",
    "Discount Taken": "D05",
}

# Deterministic first pass: plain substring keyword matching, case-insensitive.
REASON_KEYWORDS = {
    "Pricing Issue": ["price", "pricing", "rate discrepancy", "unit cost"],
    "Freight Claim": ["freight", "shipping", "carrier", "transport"],
    "Damage Claim": ["damage", "damaged", "broken", "defect"],
    "Tax Difference": ["tax", "vat", "gst"],
    "Discount Taken": ["discount", "early pay", "rebate", "promo"],
}

pd.DataFrame(REASON_CODE_MAP.items(), columns=["Reason", "Code"])

,Reason,Code
0,Pricing Issue,D01
1,Freight Claim,D02
2,Damage Claim,D03
3,Tax Difference,D04
4,Discount Taken,D05


## 3. Sample remittance / cash application lines

Ten lines chosen to exercise every branch: an exact match, one deduction per supported reason
(clean keyword text), one small variance inside tolerance (auto write-off), one messy free-text
remark that keyword matching can't confidently classify (LLM fallback), one deduction with **no**
remark at all (routed for follow-up), and one overpayment.

In [4]:
remittance_lines = [
    dict(invoice_number="INV-2001", customer_name="Nova Textiles",     invoice_amount=10000.00, payment_received=9500.00,
         remark="Short paid due to pricing discrepancy on unit rate"),
    dict(invoice_number="INV-2002", customer_name="Orion Metals",      invoice_amount=8000.00,  payment_received=7700.00,
         remark="Freight cost deducted per contract terms"),
    dict(invoice_number="INV-2003", customer_name="Pinnacle Foods",    invoice_amount=5000.00,  payment_received=4850.00,
         remark="Goods arrived damaged during transit"),
    dict(invoice_number="INV-2004", customer_name="Quantum Devices",   invoice_amount=12000.00, payment_received=11760.00,
         remark="VAT adjustment per local regulation"),
    dict(invoice_number="INV-2005", customer_name="Nova Textiles",     invoice_amount=6000.00,  payment_received=5880.00,
         remark="2% early payment discount applied per terms"),
    dict(invoice_number="INV-2006", customer_name="Riverbend Supplies", invoice_amount=3000.00, payment_received=2980.00,
         remark="Small rounding difference"),
    dict(invoice_number="INV-2007", customer_name="Silverline Corp",   invoice_amount=9000.00,  payment_received=9000.00,
         remark=None),
    dict(invoice_number="INV-2008", customer_name="Titan Freight",     invoice_amount=7000.00,  payment_received=6600.00,
         remark="Deduction taken - see attached backup, contract clause 4.2 applies re: logistics surcharge waiver"),
    dict(invoice_number="INV-2009", customer_name="Umbra Analytics",   invoice_amount=4000.00,  payment_received=3550.00,
         remark=None),
    dict(invoice_number="INV-2010", customer_name="Vertex Robotics",   invoice_amount=5000.00,  payment_received=5200.00,
         remark="Overpayment - customer paid duplicate invoice by mistake"),
]

pd.DataFrame(remittance_lines)

,invoice_number,customer_name,invoice_amount,payment_received,remark
0,INV-2001,Nova Textiles,10000.0,9500.0,Short paid due to pricing discrepancy on unit ...
1,INV-2002,Orion Metals,8000.0,7700.0,Freight cost deducted per contract terms
2,INV-2003,Pinnacle Foods,5000.0,4850.0,Goods arrived damaged during transit
3,INV-2004,Quantum Devices,12000.0,11760.0,VAT adjustment per local regulation
4,INV-2005,Nova Textiles,6000.0,5880.0,2% early payment discount applied per terms
5,INV-2006,Riverbend Supplies,3000.0,2980.0,Small rounding difference
6,INV-2007,Silverline Corp,9000.0,9000.0,NaN
7,INV-2008,Titan Freight,7000.0,6600.0,"Deduction taken - see attached backup, contrac..."
8,INV-2009,Umbra Analytics,4000.0,3550.0,NaN
9,INV-2010,Vertex Robotics,5000.0,5200.0,Overpayment - customer paid duplicate invoice ...


## 4. Building blocks

Each Given/When/Then from the brief maps to one small function, kept separate so they can be
tested and reused independently (e.g. `apply_partial_cash` doesn't care *why* there's a
deduction).

In [5]:
def identify_short_payment(invoice_amount: float, payment_received: float) -> float:
    """Given an invoice amount, When payment is less, Then return the deduction (positive = short
    payment, negative = overpayment, zero = paid in full)."""
    return round(invoice_amount - payment_received, 2)


def apply_partial_cash(invoice_amount: float, payment_received: float) -> dict:
    """Close the paid portion, leave any shortfall open for investigation."""
    deduction = identify_short_payment(invoice_amount, payment_received)
    closed_amount = round(min(invoice_amount, payment_received), 2)
    open_deduction_amount = round(max(deduction, 0.0), 2)
    return dict(closed_amount=closed_amount, open_deduction_amount=open_deduction_amount, deduction=deduction)


def _keyword_match(remark: str) -> list:
    remark_lower = remark.lower()
    return [reason for reason, kws in REASON_KEYWORDS.items() if any(k in remark_lower for k in kws)]


def _llm_classify_reason(remark: str) -> tuple:
    """Fallback for remarks the keyword pass can't confidently classify. Asks the Foundry gpt-5
    agent for strict JSON so this stays machine-parseable."""
    categories = ", ".join(REASON_CODE_MAP)
    prompt = (
        f"Classify this remittance deduction remark into exactly one of: {categories}, or "
        "\"Unknown\" if none clearly apply.\n\n"
        f'Remark: "{remark}"\n\n'
        "Respond with ONLY a compact JSON object, no other text, shaped exactly like: "
        '{"reason": "<category>", "confidence": <0.0-1.0>}'
    )
    text = ask_agent_text(prompt)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return "Unknown", 0.0
    try:
        data = json.loads(match.group(0))
        reason = data.get("reason", "Unknown")
        if reason not in REASON_CODE_MAP:
            reason = "Unknown"
        return reason, float(data.get("confidence", 0.0))
    except (json.JSONDecodeError, TypeError, ValueError):
        return "Unknown", 0.0


def classify_reason(remark) -> tuple:
    """Assign Deduction Reason Code: returns (reason, reason_code, source).
    source is "none" (no remark), "keyword" (deterministic), or "llm" (free-text fallback)."""
    if not remark:
        return "Unknown", None, "none"

    hits = _keyword_match(remark)
    if len(hits) == 1:
        reason = hits[0]
        return reason, REASON_CODE_MAP[reason], "keyword"

    reason, _confidence = _llm_classify_reason(remark)
    code = REASON_CODE_MAP.get(reason)
    return reason, code, "llm"


_deduction_ids = itertools.count(1)


def create_deduction_record(invoice_number, customer_name, deduction_amount, reason, reason_code, status) -> dict:
    return dict(
        deduction_id=f"DED-{next(_deduction_ids):04d}",
        invoice_number=invoice_number,
        customer_name=customer_name,
        deduction_amount=deduction_amount,
        reason=reason,
        reason_code=reason_code,
        status=status,
        created_date=str(date.today()),
    )

## 5. End-to-end pipeline

Mirrors Process Flow steps 3–9 from the brief for a single remittance line: match invoice (already
resolved in this sample data) → compare amounts → identify deduction/over-payment → capture reason
→ apply payment → create deduction record → assign reason code → auto write-off if within
tolerance, else route for resolution.

When the reason can't be determined at all (no remark, and — since there's no text — no keyword or
LLM classification to run), the pipeline calls the Foundry agent to draft the customer follow-up
email itself, which is Process Flow step **"2.1 Reach out to collector/customer if not
received"**.

In [6]:
def process_remittance_line(line: dict, tolerance: float = 50.00) -> dict:
    invoice_amount = line["invoice_amount"]
    payment_received = line["payment_received"]
    deduction_amount = identify_short_payment(invoice_amount, payment_received)
    cash = apply_partial_cash(invoice_amount, payment_received)

    result = dict(
        invoice_number=line["invoice_number"], customer_name=line["customer_name"],
        invoice_amount=invoice_amount, payment_received=payment_received,
        deduction_amount=deduction_amount, closed_amount=cash["closed_amount"],
        reason=None, reason_code=None, reason_source=None,
        record_type="None", status=None, deduction_id=None,
        suggested_customer_message=None,
    )

    if deduction_amount == 0:
        result["status"] = "Closed - Paid in Full"
        return result

    if deduction_amount < 0:
        result.update(record_type="Overpayment", reason="Overpayment", status="Open - Refund/Credit Review")
        rec = create_deduction_record(line["invoice_number"], line["customer_name"], deduction_amount,
                                       "Overpayment", None, result["status"])
        result["deduction_id"] = rec["deduction_id"]
        return result

    # deduction_amount > 0 -> short payment
    if abs(deduction_amount) <= tolerance:
        reason, code, source = classify_reason(line.get("remark"))
        result.update(record_type="Deduction", reason=reason, reason_code=code, reason_source=source,
                       status="Auto Written Off (Tolerance)")
        return result

    reason, code, source = classify_reason(line.get("remark"))
    result.update(record_type="Deduction", reason=reason, reason_code=code, reason_source=source)

    if reason == "Unknown":
        result["status"] = "Open - Reason Pending"
        result["suggested_customer_message"] = ask_agent_text(
            "Draft a short, polite 3-sentence email to a customer AP contact asking them to explain "
            f"the reason for a short payment of {deduction_amount:.2f} against invoice "
            f"{line['invoice_number']} (invoice amount {invoice_amount:.2f}, paid {payment_received:.2f})."
        )
    else:
        result["status"] = "Open - Pending Resolution"

    rec = create_deduction_record(line["invoice_number"], line["customer_name"], deduction_amount,
                                   reason, code, result["status"])
    result["deduction_id"] = rec["deduction_id"]
    return result


TOLERANCE = 50.00
results_df = pd.DataFrame([process_remittance_line(line, tolerance=TOLERANCE) for line in remittance_lines])
results_df.drop(columns=["suggested_customer_message"])

,invoice_number,customer_name,invoice_amount,payment_received,deduction_amount,closed_amount,reason,reason_code,reason_source,record_type,status,deduction_id
0,INV-2001,Nova Textiles,10000.0,9500.0,500.0,9500.0,Pricing Issue,D01,keyword,Deduction,Open - Pending Resolution,DED-0001
1,INV-2002,Orion Metals,8000.0,7700.0,300.0,7700.0,Freight Claim,D02,keyword,Deduction,Open - Pending Resolution,DED-0002
2,INV-2003,Pinnacle Foods,5000.0,4850.0,150.0,4850.0,Damage Claim,D03,keyword,Deduction,Open - Pending Resolution,DED-0003
3,INV-2004,Quantum Devices,12000.0,11760.0,240.0,11760.0,Tax Difference,D04,keyword,Deduction,Open - Pending Resolution,DED-0004
4,INV-2005,Nova Textiles,6000.0,5880.0,120.0,5880.0,Discount Taken,D05,keyword,Deduction,Open - Pending Resolution,DED-0005
5,INV-2006,Riverbend Supplies,3000.0,2980.0,20.0,2980.0,Pricing Issue,D01,llm,Deduction,Auto Written Off (Tolerance),NaN
6,INV-2007,Silverline Corp,9000.0,9000.0,0.0,9000.0,NaN,NaN,NaN,None,Closed - Paid in Full,NaN
7,INV-2008,Titan Freight,7000.0,6600.0,400.0,6600.0,Pricing Issue,D01,llm,Deduction,Open - Pending Resolution,DED-0006
8,INV-2009,Umbra Analytics,4000.0,3550.0,450.0,3550.0,Unknown,NaN,none,Deduction,Open - Reason Pending,DED-0007
9,INV-2010,Vertex Robotics,5000.0,5200.0,-200.0,5000.0,Overpayment,NaN,NaN,Overpayment,Open - Refund/Credit Review,DED-0008


### Reason classification detail — which lines were keyword vs. LLM vs. none

In [7]:
results_df[["invoice_number", "deduction_amount", "reason", "reason_code", "reason_source", "status"]]

,invoice_number,deduction_amount,reason,reason_code,reason_source,status
0,INV-2001,500.0,Pricing Issue,D01,keyword,Open - Pending Resolution
1,INV-2002,300.0,Freight Claim,D02,keyword,Open - Pending Resolution
2,INV-2003,150.0,Damage Claim,D03,keyword,Open - Pending Resolution
3,INV-2004,240.0,Tax Difference,D04,keyword,Open - Pending Resolution
4,INV-2005,120.0,Discount Taken,D05,keyword,Open - Pending Resolution
5,INV-2006,20.0,Pricing Issue,D01,llm,Auto Written Off (Tolerance)
6,INV-2007,0.0,NaN,NaN,NaN,Closed - Paid in Full
7,INV-2008,400.0,Pricing Issue,D01,llm,Open - Pending Resolution
8,INV-2009,450.0,Unknown,NaN,none,Open - Reason Pending
9,INV-2010,-200.0,Overpayment,NaN,NaN,Open - Refund/Credit Review


### Customer follow-up drafted by the Foundry agent (reason unknown, no remark at all)

In [8]:
for _, row in results_df[results_df["suggested_customer_message"].notna()].iterrows():
    print(f"--- {row['invoice_number']} ({row['customer_name']}) — deduction {row['deduction_amount']:.2f} ---")
    print(row["suggested_customer_message"])
    print()

--- INV-2009 (Umbra Analytics) — deduction 450.00 ---
Hello [AP Contact Name],

We received a payment of 3,550.00 for invoice INV-2009, which is 450.00 short of the invoiced total of 4,000.00. Could you please explain the reason for the short payment and share any remittance details so we can update our records? Thank you for your help.

Best regards,
[Your Name]
[Your Company]
[Your Contact Information]



## 6. Portfolio summary (agent-generated)

Same pattern as Capstone 01: the deterministic pipeline produces the structured `results_df`, and
the agent is only asked to narrate it for a human reader.

In [9]:
summary_cols = ["invoice_number", "customer_name", "deduction_amount", "reason", "status"]
summary_prompt = (
    "Summarize this deduction-identification run for an AR manager in 3-4 sentences. "
    "Call out total deduction value, how many are auto-written-off vs open, and any invoice still "
    "waiting on a reason from the customer.\n\n"
    + results_df[summary_cols].to_string(index=False)
)
print(ask_agent_text(summary_prompt))

Total identified deduction value is 2,180. One deduction (INV-2006, 20) was auto written off under tolerance, and eight items remain open across Pending Resolution, Reason Pending, and Refund/Credit Review statuses. INV-2009 (Umbra Analytics, 450) is still waiting on a reason from the customer. Note: INV-2010 reflects a -200 overpayment under Refund/Credit Review and is not included in the total deduction value.


## 7. Solution summary — requirement → implementation

| Requirement | Implementation | Why |
|---|---|---|
| Identify Short Payment | `identify_short_payment()` | Direct `invoice_amount - payment_received`; sign tells short-pay vs overpayment |
| Read Deduction Reason from Remittance | `classify_reason()` — keyword pass first | Deterministic for the common, clearly-worded cases |
| ...free-text fallback | `_llm_classify_reason()` calling the Foundry `gpt-5` agent, strict-JSON prompt | Handles messy real-world remittance wording without hand-written regexes for every phrasing |
| Create Deduction Record | `create_deduction_record()` | One record per open deduction/overpayment, with an ID an analyst can track |
| Apply Partial Cash | `apply_partial_cash()` | Closes the paid portion, leaves only the shortfall (`open_deduction_amount`) outstanding |
| Assign Deduction Reason Code | `REASON_CODE_MAP` (D01–D05), applied inside `classify_reason()` | Matches the brief's sample codes exactly |
| Auto Match Based on Tolerance | `tolerance` parameter in `process_remittance_line()` | Line INV-2006 (20 vs. 50 tolerance) auto-writes-off instead of creating a deduction |
| "reach out to collector/customer" (Process Flow 2.1) | Agent-drafted `suggested_customer_message` when reason is `Unknown` | Only triggered when there's truly nothing to classify from (no remark) |

**Design choices worth calling out:**
- Reason classification tries the free, instant, 100%-reproducible keyword pass before ever
  calling the model — the LLM is reserved for the minority of remarks that need real language
  understanding (line INV-2008).
- Overpayment isn't explicitly one of the six end states in the brief, but the Process Flow
  explicitly calls out "Identify deduction difference/**Over Payment**", so it's modeled as its own
  `record_type` rather than forced into the deduction shape.
- `classify_reason()` and `apply_partial_cash()` are pure functions with no I/O — only the two
  spots that genuinely need language understanding (unclear remark, drafting an email) touch the
  network, which keeps the rest of the pipeline fast and unit-testable.
